# Head Sayısı vs Layer Sayısı - Benchmark Analizi

Bu notebook, **attention head sayısı** ile **transformer layer sayısının** model performansına etkisini karşılaştırmalı olarak analiz eder.

## İçerik
1. Model konfigürasyonları oluşturma
2. FLOPs hesaplama
3. Parametre sayısı karşılaştırması
4. Memory kullanımı analizi
5. Inference hızı benchmark'ı
6. Görselleştirmeler

---

## 1. Kütüphaneler ve Model Tanımları

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from thop import profile
import warnings
warnings.filterwarnings('ignore')

# GPT Model sınıflarını import et
import sys
sys.path.append('01_main-chapter-code')
from gpt import GPTModel, TransformerBlock, MultiHeadAttention, LayerNorm

---

## 2. Model Konfigürasyonları

In [ ]:
# Temel konfigürasyon
BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,  # Benchmark için dropout kapatıldı
    "qkv_bias": True
}

# Aynı parametre sayısına sahip farklı konfigürasyonlar
# Toplam yaklaşık: 124M parametre

# Senaryo 1: Orjinal GPT-2 small (12 layer, 12 head)
config_high_layers = {
    **BASE_CONFIG,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12
}

# Senaryo 2: Daha az layer, daha fazla head (6 layer, 24 head)
config_high_heads = {
    **BASE_CONFIG,
    "emb_dim": 768,
    "n_heads": 24,
    "n_layers": 6
}

# Senaryo 3: Orta seviye (8 layer, 16 head)
config_balanced = {
    **BASE_CONFIG,
    "emb_dim": 768,
    "n_heads": 16,
    "n_layers": 8
}

# Senaryo 4: Çok az layer, çok fazla head (4 layer, 32 head)
config_very_high_heads = {
    **BASE_CONFIG,
    "emb_dim": 768,
    "n_heads": 32,
    "n_layers": 4
}

# Senaryo 5: Çok fazla layer, az head (24 layer, 6 head)
config_very_high_layers = {
    **BASE_CONFIG,
    "emb_dim": 768,
    "n_heads": 6,
    "n_layers": 24
}

configs = {
    "12L × 12H (Orijinal)": config_high_layers,
    "8L × 16H (Dengeli)": config_balanced,
    "6L × 24H (Yüksek Head)": config_high_heads,
    "4L × 32H (Çok Yüksek Head)": config_very_high_heads,
    "24L × 6H (Çok Yüksek Layer)": config_very_high_layers
}

print("=" * 60)
print("MODEL KONFİGÜRASYONLARI")
print("=" * 60)
for name, cfg in configs.items():
    print(f"\n{name}:")
    print(f"   emb_dim: {cfg['emb_dim']}, n_heads: {cfg['n_heads']}, n_layers: {cfg['n_layers']}")

---

## 3. Parametre Hesaplama Fonksiyonu

In [ ]:
def calculate_parameters(emb_dim, n_heads, n_layers, vocab_size=50257, context_length=1024):
    """Model parametrelerini hesapla"""
    
    # Embedding katmanları
    tok_emb_params = vocab_size * emb_dim
    pos_emb_params = context_length * emb_dim
    
    # Her transformer block için
    # Attention: Q, K, V projection + output projection
    qkv_params = 3 * (emb_dim * emb_dim)
    out_proj_params = emb_dim * emb_dim
    attention_params = qkv_params + out_proj_params
    
    # Feed Forward: up + down projection
    ffn_up = emb_dim * (emb_dim * 4)
    ffn_down = (emb_dim * 4) * emb_dim
    ffn_params = ffn_up + ffn_down
    
    # LayerNorm (2 tane: pre-attention ve pre-FFN)
    ln_params = 2 * emb_dim
    
    # Bir block'un toplamı
    block_params = attention_params + ffn_params + ln_params
    
    # Tüm block'lar
    total_block_params = block_params * n_layers
    
    # Final LayerNorm
    final_ln_params = 2 * emb_dim
    
    # Output Head (weight tying yok)
    out_head_params = emb_dim * vocab_size
    
    # Toplam
    total = tok_emb_params + pos_emb_params + total_block_params + final_ln_params + out_head_params
    
    return {
        "token_embedding": tok_emb_params,
        "positional_embedding": pos_emb_params,
        "attention_per_block": attention_params,
        "ffn_per_block": ffn_params,
        "block_params": block_params,
        "total_block_params": total_block_params,
        "final_ln": final_ln_params,
        "output_head": out_head_params,
        "total_params": total
    }

# Tüm konfigürasyonlar için parametreleri hesapla
print("=" * 70)
print("PARAMETRE SAYISI KARŞILAŞTIRMASI")
print("=" * 70)

results = []
for name, cfg in configs.items():
    params = calculate_parameters(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'])
    results.append({
        "Konfigürasyon": name,
        "emb_dim": cfg['emb_dim'],
        "n_heads": cfg['n_heads'],
        "n_layers": cfg['n_layers'],
        "Head/Layer": cfg['n_heads'] / cfg['n_layers'],
        "Parametre (M)": params['total_params'] / 1_000_000
    })
    print(f"\n{name}:")
    print(f"   Toplam Parametre: {params['total_params']:,}")
    print(f"   Block başına:   {params['block_params']:,}")

df_params = pd.DataFrame(results)
print("\n" + "=" * 70)
print(df_params.to_string(index=False))

---

## 4. FLOPs Hesaplama

In [ ]:
def calculate_flops(emb_dim, n_heads, n_layers, seq_len, vocab_size=50257):
    """
    Forward pass FLOPs hesaplama
    
    Not: Bu yaklaşık bir hesaptır
    """
    batch_size = 1
    
    # Token + Positional Embedding
    embedding_flops = 2 * batch_size * seq_len * vocab_size * emb_dim  # token embedding
    
    # Attention başına FLOPs (her layer için)
    # Q, K, V projections
    qkv_flops = 3 * (batch_size * seq_len * emb_dim * emb_dim * 2)  # *2 for matmul
    
    # Attention scores (Q @ K^T)
    attn_scores_flops = 2 * (batch_size * n_heads * seq_len * seq_len * (emb_dim // n_heads))
    
    # Softmax
    softmax_flops = batch_size * n_heads * seq_len * seq_len * 5  # approximate
    
    # Attention @ values
    attn_values_flops = 2 * (batch_size * n_heads * seq_len * seq_len * (emb_dim // n_heads))
    
    # Output projection
    out_proj_flops = batch_size * seq_len * emb_dim * emb_dim * 2
    
    # Bir attention layer'ın toplamı
    attn_layer_flops = qkv_flops + attn_scores_flops + softmax_flops + attn_values_flops + out_proj_flops
    
    # Feed Forward Network
    ffn_up_flops = batch_size * seq_len * emb_dim * (emb_dim * 4) * 2
    ffn_down_flops = batch_size * seq_len * (emb_dim * 4) * emb_dim * 2
    ffn_layer_flops = ffn_up_flops + ffn_down_flops
    
    # LayerNorm (2 tane)
    ln_flops = 2 * batch_size * seq_len * emb_dim * 10  # approximate
    
    # Bir transformer block
    block_flops = attn_layer_flops + ffn_layer_flops + ln_flops
    
    # Tüm layer'lar
    total_block_flops = block_flops * n_layers
    
    # Output head
    output_flops = batch_size * seq_len * emb_dim * vocab_size * 2
    
    # Final LayerNorm
    final_ln_flops = batch_size * seq_len * emb_dim * 10
    
    total_flops = embedding_flops + total_block_flops + output_flops + final_ln_flops
    
    return total_flops

# Farklı sequence length'ler için FLOPs hesapla
seq_lengths = [64, 128, 256, 512, 1024]

print("=" * 70)
print("FLOPs KARŞILAŞTIRMASI (Forward Pass)")
print("=" * 70)

flops_data = []
for seq_len in seq_lengths:
    row = {"Sequence Length": seq_len}
    for name, cfg in configs.items():
        flops = calculate_flops(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], seq_len)
        row[name] = flops / 1e12  # TFLOPs
    flops_data.append(row)

df_flops = pd.DataFrame(flops_data)
print(df_flops.to_string(index=False))

# En yaygın sequence length için detay
print("\n" + "=" * 70)
print("FLOPs Detayı (seq_len=512)")
print("=" * 70)
for name, cfg in configs.items():
    flops = calculate_flops(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], 512)
    print(f"{name}: {flops/1e12:.2f} TFLOPs")

---

## 5. Memory Kullanımı Analizi

In [ ]:
def calculate_memory(emb_dim, n_heads, n_layers, seq_len, vocab_size=50257, precision="float32"):
    """
    Model memory gereksinimini hesapla
    
    precision: float32, float16, bfloat16, int8
    """
    params = calculate_parameters(emb_dim, n_heads, n_layers, vocab_size)
    total_params = params['total_params']
    
    # Precision'a göre byte hesapla
    bytes_map = {
        "float32": 4,
        "float16": 2,
        "bfloat16": 2,
        "int8": 1
    }
    bytes_per_param = bytes_map.get(precision, 4)
    
    # Model memory
    model_memory = total_params * bytes_per_param
    
    # Training için: gradients
    gradient_memory = total_params * bytes_per_param
    
    # Training için: optimizer state (AdamW)
    optimizer_memory = total_params * bytes_per_param * 2
    
    # KV Cache (inference için)
    # Her layer için: 2 * batch * seq * n_heads * head_dim
    head_dim = emb_dim // n_heads
    kv_cache_memory = 2 * seq_len * n_layers * n_heads * head_dim * bytes_per_param
    
    # Activation'lar (yaklaşık)
    activation_memory = batch_size * seq_len * emb_dim * n_layers * bytes_per_param * 4
    
    return {
        "model_mb": model_memory / (1024**2),
        "gradient_mb": gradient_memory / (1024**2),
        "optimizer_mb": optimizer_memory / (1024**2),
        "kv_cache_mb": kv_cache_memory / (1024**2),
        "activation_mb": activation_memory / (1024**2),
        "inference_total_mb": (model_memory + kv_cache_memory) / (1024**2),
        "training_total_mb": (model_memory + gradient_memory + optimizer_memory + activation_memory) / (1024**2)
    }

batch_size = 1
seq_len = 512

print("=" * 70)
print("MEMORY KULLANIMI (FP32, seq_len=512)")
print("=" * 70)

memory_data = []
for name, cfg in configs.items():
    mem = calculate_memory(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], seq_len)
    memory_data.append({
        "Konfigürasyon": name,
        "Model (MB)": f"{mem['model_mb']:.1f}",
        "KV Cache (MB)": f"{mem['kv_cache_mb']:.1f}",
        "Inference Toplam (MB)": f"{mem['inference_total_mb']:.1f}",
        "Training Toplam (GB)": f"{mem['training_total_mb']/1024:.2f}"
    })

df_memory = pd.DataFrame(memory_data)
print(df_memory.to_string(index=False))

# Farklı precision'lar için karşılaştırma
print("\n" + "=" * 70)
print("FARKLI PRECISION'LARDA MEMORY (Orijinal 12L×12H)")
print("=" * 70)
cfg = configs["12L × 12H (Orijinal)"]
for prec in ["float32", "float16", "bfloat16", "int8"]:
    mem = calculate_memory(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], seq_len, precision=prec)
    print(f"{prec:12s}: Model={mem['model_mb']:6.1f}MB, KV Cache={mem['kv_cache_mb']:6.1f}MB")

---

## 6. Inference Hızı Benchmark'ı

In [ ]:
def benchmark_inference(config, seq_len=128, num_iterations=50, warmup=10):
    """
    Model inference hızını ölç
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Modeli oluştur
    model = GPTModel(config).to(device)
    model.eval()
    
    # Input oluştur
    input_ids = torch.randint(0, config['vocab_size'], (1, seq_len), device=device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input_ids)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_iterations):
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            
            logits = model(input_ids)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            end = time.perf_counter()
            times.append(end - start)
    
    return {
        "mean_ms": np.mean(times) * 1000,
        "std_ms": np.std(times) * 1000,
        "min_ms": np.min(times) * 1000,
        "max_ms": np.max(times) * 1000,
        "tokens_per_sec": seq_len / np.mean(times)
    }

# Benchmark çalıştır
device_name = "CUDA" if torch.cuda.is_available() else "CPU"
print("=" * 70)
print(f"INFERENCE HIZI BENCHMARK ({device_name})")
print("=" * 70)

seq_len = 128
benchmark_results = []

for name, cfg in configs.items():
    print(f"\n{name} ölçülüyor...")
    result = benchmark_inference(cfg, seq_len=seq_len)
    benchmark_results.append({
        "Konfigürasyon": name,
        "Ortalama (ms)": f"{result['mean_ms']:.2f}",
        "Std (ms)": f"{result['std_ms']:.2f}",
        "Min (ms)": f"{result['min_ms']:.2f}",
        "Max (ms)": f"{result['max_ms']:.2f}",
        "Tokens/s": f"{result['tokens_per_sec']:.0f}"
    })

df_benchmark = pd.DataFrame(benchmark_results)
print("\n" + "=" * 70)
print(df_benchmark.to_string(index=False))

---

## 7. Görselleştirmeler

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Parametre Karşılaştırması
ax1 = axes[0, 0]
config_names = list(configs.keys())
param_values = [calculate_parameters(c['emb_dim'], c['n_heads'], c['n_layers'])['total_params']/1e6 
                for c in configs.values()]
bars1 = ax1.bar(config_names, param_values, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B'])
ax1.set_ylabel('Parametre Sayısı (Milyon)')
ax1.set_title('Parametre Sayısı Karşılaştırması')
ax1.tick_params(axis='x', rotation=45)
for bar, val in zip(bars1, param_values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1f}M', 
             ha='center', va='bottom', fontsize=9)

# 2. FLOPs Karşılaştırması
ax2 = axes[0, 1]
seq_len_test = 512
flops_values = [calculate_flops(c['emb_dim'], c['n_heads'], c['n_layers'], seq_len_test)/1e12 
                for c in configs.values()]
bars2 = ax2.bar(config_names, flops_values, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B'])
ax2.set_ylabel('FLOPs (Trilyon)')
ax2.set_title(f'FLOPs Karşılaştırması (seq_len={seq_len_test})')
ax2.tick_params(axis='x', rotation=45)
for bar, val in zip(bars2, flops_values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'{val:.2f}T', 
             ha='center', va='bottom', fontsize=9)

# 3. Memory Karşılaştırması
ax3 = axes[1, 0]
memory_values = [calculate_memory(c['emb_dim'], c['n_heads'], c['n_layers'], 512)['kv_cache_mb'] 
                 for c in configs.values()]
bars3 = ax3.bar(config_names, memory_values, color=['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B'])
ax3.set_ylabel('KV Cache Memory (MB)')
ax3.set_title('KV Cache Memory Karşılaştırması (seq_len=512)')
ax3.tick_params(axis='x', rotation=45)
for bar, val in zip(bars3, memory_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.0f}MB', 
             ha='center', va='bottom', fontsize=9)

# 4. Head vs Layer Etkisi
ax4 = axes[1, 1]
layers = [c['n_layers'] for c in configs.values()]
heads = [c['n_heads'] for c in configs.values()]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
scatter = ax4.scatter(layers, heads, s=[v*10 for v in param_values], c=colors, alpha=0.7, edgecolors='black')
for i, name in enumerate(config_names):
    ax4.annotate(name.split('×')[0].strip(), (layers[i], heads[i]), 
                 textcoords="offset points", xytext=(5,5), fontsize=8)
ax4.set_xlabel('Layer Sayısı')
ax4.set_ylabel('Head Sayısı')
ax4.set_title('Layer vs Head (Boyut = Parametre)')

plt.tight_layout()
plt.savefig('head_vs_layer_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGrafik 'head_vs_layer_benchmark.png' olarak kaydedildi.")

---

## 8. Özet ve Sonuçlar

In [ ]:
def print_summary():
    summary = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                    BENCHMARK SONUÇLARI ÖZETİ                                 ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  📊 HEAD SAYISI ARTTIRMA:                                                    ║
║     ✅ Daha paralel hesaplama (GPU'da daha verimli)                          ║
║     ✅ Daha düşük latency (sıralı işlem azalır)                              ║
║     ✅ Her head farklı özellik öğrenir (diversity)                          ║
║     ❌ KV Cache memory artar (her head'in ayrı K, V saklanır)               ║
║     ❌ Çok fazla head = paralelizasyon zorlaşır                            ║
║                                                                              ║
║  📊 LAYER SAYISI ARTTIRMA:                                                   ║
║     ✅ Daha derin temsiller (soyut öğrenme)                                  ║
║     ✅ Daha iyi gradient flow (residual connections)                        ║
║     ✅ Uzun vadede daha iyi performans                                       ║
║     ❌ Inference daha yavaş (her token için daha fazla layer)               ║
║     ❌ Daha fazla sequential computation                                    ║
║                                                                              ║
║  🎯 PRATİK ÖNERİLER:                                                        ║
║                                                                              ║
║     • Hızlı inference → Daha az layer, daha fazla head                      ║
║     • Düşük memory → GQA (Grouped-Query Attention) kullan                   ║
║     • Uzun context → Layer artır + Sliding Window Attention                 ║
║     • En iyi kalite → Hem layer hem head artır + daha büyük embedding      ║
║                                                                              ║
║  💡 MODERN LLM'LER NE KULLANIYOR:                                           ║
║                                                                              ║
║     • Llama 3:  32-80 layer,  24-32 head                                    ║
║     • GPT-4:   ~120 layer,   ~96 head  (bilinmiyor kesin)                  ║
║     • Mixtral: 32 layer,    8 head + MoE (8 expert)                        ║
║     • DeepSeek: 60 layer,   64 head + MLA                                   ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
    """
    print(summary)

print_summary()

---

## 9. Detaylı Tablo

In [ ]:
# Tüm sonuçları birleştirilmiş tablo olarak göster
print("=" * 90)
print("TAM KARŞILAŞTIRMA TABLOSU")
print("=" * 90)

final_data = []
for name, cfg in configs.items():
    params = calculate_parameters(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'])
    flops_512 = calculate_flops(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], 512)
    mem = calculate_memory(cfg['emb_dim'], cfg['n_heads'], cfg['n_layers'], 512)
    
    final_data.append({
        "Konfigürasyon": name,
        "Layers": cfg['n_layers'],
        "Heads": cfg['n_heads'],
        "Head/Layer": f"{cfg['n_heads']/cfg['n_layers']:.2f}",
        "Param (M)": f"{params['total_params']/1e6:.1f}",
        "FLOPs (T)": f"{flops_512/1e12:.2f}",
        "Model (MB)": f"{mem['model_mb']:.0f}",
        "KV Cache (MB)": f"{mem['kv_cache_mb']:.0f}"
    })

df_final = pd.DataFrame(final_data)
print(df_final.to_string(index=False))

# Karşılaştırma metrikleri
print("\n" + "=" * 90)
print("ANA ÇIKARIMLAR")
print("=" * 90)

# En hızlı (en az FLOPs)
flops_list = [(name, calculate_flops(c['emb_dim'], c['n_heads'], c['n_layers'], 512)/1e12) 
               for name, c in configs.items()]
fastest = min(flops_list, key=lambda x: x[1])
print(f"\n🚀 En az FLOPs: {fastest[0]} ({fastest[1]:.2f}T)")

# En düşük KV cache
kv_list = [(name, calculate_memory(c['emb_dim'], c['n_heads'], c['n_layers'], 512)['kv_cache_mb']) 
            for c in configs.values() for name, c2 in configs.items() if c is c2]
# Düzelt
kv_list = []
for name, c in configs.items():
    mem = calculate_memory(c['emb_dim'], c['n_heads'], c['n_layers'], 512)
    kv_list.append((name, mem['kv_cache_mb']))

lowest_kv = min(kv_list, key=lambda x: x[1])
print(f"💾 En düşük KV Cache: {lowest_kv[0]} ({lowest_kv[1]:.0f}MB)")

# En yüksek paralellik (head/layer oranı)
ratio_list = [(name, c['n_heads']/c['n_layers']) for name, c in configs.items()]
highest_ratio = max(ratio_list, key=lambda x: x[1])
print(f"⚡ En yüksek paralellik: {highest_ratio[0]} (Head/Layer: {highest_ratio[1]:.2f})")

---

## Referanslar

- Sebastian Raschka - Build a Large Language Model From Scratch
- Bu notebook, Bölüm 4.6'daki GPT model incelemesinin bir parçasıdır
- Detaylı attention analizi için: `../04_gqa/` klasörü